# Choi2025-ORL-OusvMC

### Title:
Exact simulation scheme for the Ornstein–Uhlenbeck driven stochastic volatility model with the Karhunen–Loève expansions

### Author: 
* Jaehyuk Choi ([@jaehyukchoi](https://github.com/jaehyukchoi))

### Abstract:
This study proposes a fast exact simulation scheme for the Ornstein–Uhlenbeck driven stochastic volatility model. With the Karhunen–Loève expansions, the stochastic volatility path (Ornstein–Uhlenbeck process) is expressed as a sine series, and the time integrals of volatility and variance are analytically derived as infinite series of independent normal random variables. The new method is several hundred times faster than the existing method using numerical transform inversion. The simulation variance is further reduced with conditional simulation and the control variate.

### Link
* DOI: [10.1016/j.orl.2025.107280](https://doi.org/10.1016/j.orl.2025.107280)
* arXiv: [2402.09243](https://arxiv.org/abs/2402.09243)

In [1]:
import numpy as np
import time
import pandas as pd
import pyfeng as pf

### Benchmark Parameter Sets and FFT pricing

In [2]:
pf.OusvFft.init_benchmark()

,sigma,theta,vov,rho,mr,texp,spot,intr,col_name,Reference
Sheet,,,,,,,,,,
1,0.20,0.20,0.1,-0.7,4,1.0,100,0.09531,Price,Table 2 in Li and Wu (2019)
2,0.20,0.20,0.1,-0.7,4,5.0,100,0.09531,Price,Table 2 in Li and Wu (2019)
3,0.20,0.20,0.1,-0.7,4,10.0,100,0.09531,Price,Table 2 in Li and Wu (2019)
4,0.25,0.25,0.3,-0.6,8,1.0,100,0.09531,Price,Table 3 in Li and Wu (2019)
5,0.20,0.20,0.1,0.0,4,0.5,100,0.09530,Price,Table 1A in Schobel & Zhu (1999)
6,0.20,0.10,0.1,0.0,4,0.5,100,0.09530,Price,Table 1B in Schobel & Zhu (1999)
7,0.20,0.30,0.1,0.0,4,0.5,100,0.09530,Price,Table 1C in Schobel & Zhu (1999)
8,0.15,0.00,0.1,0.5,4,0.5,100,0.09530,Price,Table 2E in Schobel & Zhu (1999)
9,0.15,0.00,0.1,0.0,4,0.5,100,0.09530,Price,Table 2D/F in Schobel & Zhu (1999)


In [3]:
sheet_no = 1
m1, p, rv = pf.OusvFft.init_benchmark(sheet_no)
np.round(m1.price(**rv['args_pricing']), 6) - rv['val']

array([0.])

## Reproduction of Table 1/2/3 (Sheet 1/2/3)

In [4]:
params = {'sigma': 0.2, 'theta': 0.2, 'mr': 4, 'vov':0.1, 'rho':-0.7, 'intr': 0.09531}
spot = 100
strike = 100
texp_arr = [1, 5, 10] #
n_sin_arr2 = [[2, 4, 6], [4, 6, 8], [6, 8, 10]]
p_exact_arr = [13.21493, 40.79773, 62.76312] # 

### Quick Test

In [5]:
m = pf.OusvMcChoi2025KL(**params)
m.set_num_params(n_path = 10000, dt=None, rn_seed=12345, n_sin=2)
m.price(strike, spot, 5.0)

np.float64(40.68576971967935)

### Full Simulation

In [6]:
m = pf.OusvMcChoi2025KL(**params)

In [7]:
n_path = 160000
n_run = int(2560000 / n_path)

i = 2 # 0: T=1, 1: T=5, 2: T=10
texp = texp_arr[i]
n_sin_arr = n_sin_arr2[i]
p_exact = p_exact_arr[i]

In [8]:
p = np.zeros((n_run, len(n_sin_arr), 3), dtype=np.float32)

for j, n_sin in enumerate(n_sin_arr):
    m.set_num_params(n_path = n_path, dt=None, rn_seed=12345, n_sin=n_sin)
    m.correct_fwd = False
    for k in range(n_run):
        #print(f'texp={texp}, n_sin={n_sin}')
        p[k, j, 1] = m.price(strike, spot, texp) - p_exact
        p[k, j, 0] = m.result['spot error']*spot

for j, n_sin in enumerate(n_sin_arr):
    m.set_num_params(n_path = n_path, dt=None, rn_seed=12345, n_sin=n_sin)
    m.correct_fwd = True
    for k in range(n_run):
        #print(f'texp={texp}, n_sin={n_sin}')
        p[k, j, 2] = m.price(strike, spot, texp) - p_exact

In [9]:
p_mean = np.mean(p, axis=0)*10000
p_std = np.std(p, axis=0)*100

In [10]:
df = pd.DataFrame({
    'Texp': texp, 'N Path': n_path, 
    'N Sin': n_sin_arr, 
    'Spot Bias ($\times 10^{-4}$)': p_mean[:,0], 'Spot RMSE ($\times 10^{-2}$)': p_std[:,0], 
    'Opt Bias ($\times 10^{-4}$)': p_mean[:,1], 'Opt RMSE ($\times 10^{-2}$)': p_std[:,1], 
    'OptCv Bias ($\times 10^{-4}$)': p_mean[:,2], 'OptCv RMSE ($\times 10^{-2}$)': p_std[:,2], 
})

In [11]:
df

,Texp,N Path,N Sin,Spot Bias ($\times 10^{-4}$),Spot RMSE ($\times 10^{-2}$),Opt Bias ($\times 10^{-4}$),Opt RMSE ($\times 10^{-2}$),OptCv Bias ($\times 10^{-4}$),OptCv RMSE ($\times 10^{-2}$)
0,10,160000,6,-94.315430,5.403560,-124.540878,5.752529,-33.598656,0.680316
1,10,160000,8,-5.970513,5.299690,-20.480627,5.743393,-14.688261,0.749083
2,10,160000,10,190.383698,3.253802,199.089706,3.687629,15.586046,0.638453
